In [ ]:
import os
from typing import Any

import matplotlib.pyplot as plt
import optuna
import pandas as pd
from common import (
    METRIC_TO_LABEL,
    get_color,
    get_marker,
    get_method_label,
)

db = optuna.storages.RDBStorage(os.environ.get("OPTUNA_STORAGE"))

FIG_WIDTH_PT = 397
PT_PER_INCH = 72.27
FIG_WIDTH_IN = FIG_WIDTH_PT / PT_PER_INCH
METHODS = ["ball", "tball", "tball-mnd"]


def rename(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(
        columns={
            "user_attrs_avg_acc": "accuracy_seen_avg",
            "user_attrs_avg_ece": "ece_seen_avg",
            "params_strategy.beta": "beta",
            "params_peft.r": "rank",
            "params_peft.rank": "rank",
            "params_strategy.test_samples": "samples",
        }
    )


def load_sensitivity_df(study_name: str) -> pd.DataFrame:
    studies = {
        method: optuna.load_study(
            study_name=f"bayescl/{study_name}/imagenetr/{method}",
            storage=db,
        )
        for method in METHODS
    }
    df = pd.concat(
        [rename(study.trials_dataframe()) for study in studies.values()],
        keys=list(studies.keys()),
    )
    df["score"] = (df["accuracy_seen_avg"] + (1 - df["ece_seen_avg"])) / 2
    return df.reset_index(level=0).rename(columns={"level_0": "method"})


def preprocess_for_plot(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    return df[["method", x_col, y_col]].dropna().sort_values(["method", x_col]).copy()


def plot_panel(
    ax: Any,
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    use_log2_x: bool = False,
    title: str | None = None,
    show_ylabel: bool = True,
) -> None:
    plot_df = preprocess_for_plot(df, x_col=x_col, y_col=y_col)
    for method, method_df in plot_df.groupby("method", sort=False):
        # Aggregate by x_col, taking the mean of y_col
        method_stats = method_df.groupby(x_col, as_index=False).aggregate(
            {y_col: ["mean", "std"]}
        )
        mean = method_stats[y_col]["mean"]
        std = method_stats[y_col]["std"].fillna(0.0)
        lower = mean - std
        upper = mean + std

        ax.plot(
            method_stats[x_col],
            mean,
            marker=get_marker(method),
            markersize=3.5,
            linewidth=1.2,
            color=get_color(method),
            label=get_method_label(method),
            # linestyle=get_linestyle(method),
        )

        ax.fill_between(
            method_stats[x_col],
            lower,
            upper,
            color=get_color(method),
            alpha=0.2,
            linewidth=0,
        )

    if use_log2_x:
        ax.set_xscale("log", base=2)
    ax.set_xlabel(x_label)
    ax.set_ylabel(METRIC_TO_LABEL[y_col] if show_ylabel else "")
    if title is not None:
        ax.set_title(title)

## Sensitivity Analysis: Beta

In [ ]:
df_beta = load_sensitivity_df("01-sensitivity-beta")
df_beta.head()

## Sensitivity Analysis: Rank

In [ ]:
df_rank = load_sensitivity_df("01-sensitivity-rank")
df_rank.head()

In [ ]:
df_samples = None
df_samples = load_sensitivity_df("01-sensitivity-samples")
df_samples.head()

In [ ]:
from pathlib import Path

import scienceplots

if scienceplots is not None:
    plt.style.use(["science", "nature"])

# sns.set_context("paper", font_scale=0.75)
# sns.set_style("ticks")
fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(FIG_WIDTH_IN, FIG_WIDTH_IN / (2**0.5)),
    sharex="col",
    sharey="row",
    # gridspec_kw={"hspace": 0.0, "wspace": 0.25},
)

column_specs = [
    ("beta", df_beta, "beta", False),
    ("rank", df_rank, "rank", True),
    ("samples", df_samples, "samples", True),
]

for col_idx, (_, df_in, x_col, use_log2_x) in enumerate(column_specs):
    plot_panel(
        axes[0, col_idx],
        df=df_in,
        x_col=x_col,
        y_col="accuracy_seen_avg",
        x_label="",
        use_log2_x=use_log2_x,
        title=None,
        show_ylabel=(col_idx == 0),
    )
    plot_panel(
        axes[1, col_idx],
        df=df_in,
        x_col=x_col,
        y_col="ece_seen_avg",
        x_label="",
        use_log2_x=use_log2_x,
        show_ylabel=(col_idx == 0),
    )
    plot_panel(
        axes[2, col_idx],
        df=df_in,
        x_col=x_col,
        y_col="score",
        x_label=x_col.title(),
        use_log2_x=use_log2_x,
        show_ylabel=(col_idx == 0),
    )

reference_style = {"linestyle": "--", "linewidth": 0.5, "zorder": -1}

for row_idx in range(3):
    axes[row_idx, 0].axvline(
        1.52,
        color=get_color("ball"),
        **reference_style,
    )
    axes[row_idx, 0].axvline(
        0.0617,
        color=get_color("tball"),
        **reference_style,
    )
    axes[row_idx, 0].axvline(
        0.0262,
        color=get_color("tball-mnd"),
        **reference_style,
    )
    axes[row_idx, 1].axvline(
        10,
        color="black",
        **reference_style,
    )
    axes[row_idx, 2].axvline(
        5,
        color="black",
        **reference_style,
    )

for ax in axes[:-1, :].flatten():
    ax.tick_params(labelbottom=False)


def robust_limits(
    series: pd.Series,
    lower_q: float = 0.01,
    upper_q: float = 0.99,
    pad_frac: float = 0.1,
) -> tuple[float, float]:
    values = series.dropna().astype(float)
    low = float(values.quantile(lower_q))
    high = float(values.quantile(upper_q))
    span = max(high - low, 1e-6)
    return low - pad_frac * span, high + pad_frac * span


acc_values = pd.concat(
    [
        df_beta["accuracy_seen_avg"],
        df_rank["accuracy_seen_avg"],
        df_samples["accuracy_seen_avg"],
    ]
)
ece_values = pd.concat(
    [df_beta["ece_seen_avg"], df_rank["ece_seen_avg"], df_samples["ece_seen_avg"]]
)
score_values = pd.concat([df_beta["score"], df_rank["score"], df_samples["score"]])

acc_low, acc_high = robust_limits(acc_values)
ece_low, ece_high = robust_limits(ece_values)
score_low, score_high = robust_limits(score_values)

acc_ylim = (max(0.0, acc_low), min(1.0, acc_high))
ece_ylim = (max(0.0, ece_low), ece_high)
score_ylim = (max(0.0, score_low), min(1.0, score_high))

# for ax in axes[0, :]:
#     ax.set_ylim(*acc_ylim)
# for ax in axes[1, :]:
#     ax.set_ylim(*ece_ylim)
# for ax in axes[2, :]:
#     ax.set_ylim(*score_ylim)

# for ax in axes.flatten():
# ax.tick_params(labelleft=True)
# ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

fig.align_ylabels(axes[:, 0])
for row_idx in range(3):
    axes[row_idx, 0].yaxis.set_label_coords(-0.25, 0.5)

handles, labels = axes[0, 0].get_legend_handles_labels()
for ax in axes.flatten():
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

axes[0, 2].legend(
    handles,
    labels,
    loc="best",
    ncol=1,
    frameon=True,
    borderaxespad=0.4,
    fancybox=False,
)

save_dir = Path("analysis/plots")
if not save_dir.exists():
    save_dir = Path("plots")
save_dir.mkdir(parents=True, exist_ok=True)

# Ttile
# fig.suptitle("Sensitivity Analysis on iCIFAR100/10", y=0.95)

fig.savefig(save_dir / "sensitivity_imagenetr.pdf", bbox_inches="tight")
fig.savefig(save_dir / "sensitivity_imagenetr.png", dpi=300, bbox_inches="tight")
plt.show()